---

## **Spin–Orbit Coupling (SOC) – Practical Requirements and Why**

#### **1. Converge k–points carefully**
SOC changes band energies and splittings in k-space; a coarse grid gives noisy or incorrect bands and DOS.

#### **2. Converge mesh cut-off carefully**
SOC depends on sharp potential gradients near nuclei; a fine real-space grid is needed for accurate energies.

#### **3. Tight density matrix tolerance (< 10⁻⁵)**
SOC energy differences are very small (meV), so loose SCF convergence can hide or distort them.

#### **4. Use relativistic pseudopotentials with NLCC**
SOC is a relativistic effect; only relativistic PPs (with core corrections) correctly include spin–orbit interactions.

#### **5. Usual basis/parameters still apply**
No special new settings are required, but all parameters must be converged more strictly than normal.

#### **6. Tutorial uses loose settings only for speed**
Low cutoffs and few k-points give quick but qualitative results; real calculations must be fully converged.

---


In [1]:
!pwd

/home/l-rishiraj/siesta-docs/work-files/tutorials/basic/magnetism/03_Fe_isolated


In [2]:
ls

fe_atom.fdf  Fe.psf  TutorialSPIN3.ipynb


In [3]:
!mkdir -p EX1

In [4]:
!cp fe_atom.fdf  Fe.psf EX1/

In [5]:
%cd EX1/

/home/l-rishiraj/siesta-docs/work-files/tutorials/basic/magnetism/03_Fe_isolated/EX1


/home/l-rishiraj/miniconda3/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
ls

fe_atom.fdf  Fe.psf


## **One scalar relativistic calculation (setting Spin polarized or Spin non-colinear)**

In [8]:
!cat fe_atom.fdf

#General system specifications
SystemName          Iron (isolated atom)
SystemLabel         Fe_atom
NumberOfAtoms       1
NumberOfSpecies     1

%block ChemicalSpeciesLabel
 1  26 Fe      # Species index, atomic number, species label
%endblock ChemicalSpeciesLabel

# Basis set definition
PAO.EnergyShift 200 meV
PAO.SplitNorm   0.15
PAO.BasisSize   DZP

# Lattice vectors
LatticeConstant  2.86 Ang
%block LatticeVectors
   10.0000    0.0000    0.0000
    0.0000   10.0000    0.0000
    0.0000    0.0000   10.0000
%endblock LatticeVectors

#Atomic coordinates
AtomicCoordinatesFormat  ScaledCartesian

%block AtomicCoordinatesAndAtomicSpecies
    0.0000    0.0000    0.0000   1           
%endblock AtomicCoordinatesAndAtomicSpecies

# Real space grid 
MeshCutoff 125.0 Ry

# K points
%block kgrid.MonkhorstPack
  1    0    0    0.
  0    1    0    0.
  0    0    1    0.
%endblock kgrid.MonkhorstPack

# Convergence of SCF 
MaxSCFIterations 500
DM.MixingWeight  0.01
DM.NumberPulay   7
DM.UseSaveDM 

In [3]:
!siesta < fe_atom.fdf > fe_atom.out

Job completed


In [26]:
ls

0_NORMAL_EXIT                Fe_atom.DM          Fe.ion
BASIS_ENTHALPY               Fe_atom.EIG         Fe.ion.nc
BASIS_HARRIS_ENTHALPY        Fe_atom.FA          Fe.ion.xml
CLOCK                        fe_atom.fdf         Fe.psf
fdf.20260208T203740.802.log  Fe_atom.HSX         FORCE_STRESS
Fe_atom.alloc                Fe_atom.KP          INPUT_TMP.58003
Fe_atom.BASIS_ENTHALPY       Fe_atom.ORB_INDX    MESSAGES
Fe_atom.bib                  fe_atom.out         NON_TRIMMED_KP_LIST
Fe_atom.BONDS                Fe_atom.STRUCT_OUT  OUTVARS.yml
Fe_atom.BONDS_FINAL          Fe_atom.XV          PARALLEL_DIST


In [11]:
!grep -A 7 "Eigenvalues (eV)" fe_atom.out


siesta: Eigenvalues (eV):
  ik is       eps
   1  1   -6.3407   -6.3407   -6.3407   -6.3367   -6.3367   -4.6726    0.3483    0.3483    0.3483   14.8134
          28.0972   28.0972   28.0976   28.0976   28.0976
   1  2   -3.8849   -2.7328   -2.7328   -2.7322   -2.7322   -2.7322    0.9691    0.9691    0.9691   15.3313
          29.5846   29.5846   29.5867   29.5867   29.5867
siesta: Fermi energy =      -2.768255 eV



 ## **Why are lower orbitals (1s, 2s, 2p, 3s, 3p) not included in SIESTA eigenvalues?**

 ### **Core vs Valence concept**

SIESTA uses pseudopotentials that separate electrons into:

### **Core (frozen)**
1s, 2s, 2p, 3s, 3p  
- very low energy  
- chemically inactive  
- replaced by pseudopotential  
- NOT explicitly calculated  

### **Valence (explicit)**
3d, 4s (and polarization 4p)  
- participate in bonding and magnetism  
- treated with basis functions  
- appear in eigenvalues  

---

### **For Fe**

Full configuration:
$[
1s^2 2s^2 2p^6 3s^2 3p^6 3d^6 4s^2
]$

Core:
$[
[Ar] = 18 \text{ electrons}
]$

Valence:
$[
3d^6 4s^2
]$

Only valence states are computed.

---

### **Consequence**

Core states:
- very deep energies (~ −100s eV)
- not printed

Valence states:
- near Fermi level
- appear in eigenvalue list

---

## Why are the energies different from the first line?

The difference comes from **exchange splitting**.

In a spin-polarized magnetic system (FM or AFM Fe):

$[
E_{\uparrow} \neq E_{\downarrow}
]$

This means spin-up and spin-down electrons experience **different effective potentials** due to exchange interactions.

---

### Interpretation of the lines

- **First line → spin ↑ (up channel)**
- **Second line → spin ↓ (down channel)**

Because of exchange splitting:
- spin ↑ bands shift **lower** in energy
- spin ↓ bands shift **higher** in energy

---

### Physical consequence

This energy separation between spin channels:

- creates unequal occupation of ↑ and ↓ states  
- produces a **net magnetic moment**

Hence, the two lines correspond to the **same orbitals**, but their energies differ due to magnetism.

### **Final takeaway**

Lower orbitals are not missing — they are **frozen inside the pseudopotential**, so only valence orbitals show up in SIESTA outputs.


## **The fully relativistic case (Spin spin-orbit)**

In [12]:
%cd ..

/home/l-rishiraj/siesta-docs/work-files/tutorials/basic/magnetism/03_Fe_isolated


In [13]:
mkdir -p EX2

In [14]:
!cp fe_atom.fdf  Fe.psf EX2/

In [15]:
%cd EX2

/home/l-rishiraj/siesta-docs/work-files/tutorials/basic/magnetism/03_Fe_isolated/EX2


In [16]:
ls

fe_atom.fdf  Fe.psf


In [17]:
!cat fe_atom.fdf

#General system specifications
SystemName          Iron (isolated atom)
SystemLabel         Fe_atom
NumberOfAtoms       1
NumberOfSpecies     1

%block ChemicalSpeciesLabel
 1  26 Fe      # Species index, atomic number, species label
%endblock ChemicalSpeciesLabel

# Basis set definition
PAO.EnergyShift 200 meV
PAO.SplitNorm   0.15
PAO.BasisSize   DZP

# Lattice vectors
LatticeConstant  2.86 Ang
%block LatticeVectors
   10.0000    0.0000    0.0000
    0.0000   10.0000    0.0000
    0.0000    0.0000   10.0000
%endblock LatticeVectors

#Atomic coordinates
AtomicCoordinatesFormat  ScaledCartesian

%block AtomicCoordinatesAndAtomicSpecies
    0.0000    0.0000    0.0000   1           
%endblock AtomicCoordinatesAndAtomicSpecies

# Real space grid 
MeshCutoff 125.0 Ry

# K points
%block kgrid.MonkhorstPack
  1    0    0    0.
  0    1    0    0.
  0    0    1    0.
%endblock kgrid.MonkhorstPack

# Convergence of SCF 
MaxSCFIterations 500
DM.MixingWeight  0.01
DM.NumberPulay   7
DM.UseSaveDM 

In [18]:
!siesta < fe_atom.fdf > fe_atom.out

Job completed


In [20]:
!grep -A 6 "Eigenvalues (eV)" fe_atom.out

siesta: Eigenvalues (eV):
ik =     1
   -6.3853   -6.3637   -6.3612   -6.3230   -6.2394   -4.6719   -3.8841   -2.7729   -2.7632   -2.7444
   -2.7051   -2.6570    0.3128    0.3360    0.3890    0.9411    0.9672    1.0090   14.8139   15.3319
   28.0752   28.0796   28.0861   28.1077   28.1444   29.5617   29.5686   29.5781   29.6035   29.6313
siesta: Fermi energy =      -2.781403 eV



---

## **Comparison of Scalar Relativistic and Spin–Orbit Coupling (SOC) Calculations in SIESTA**

We compare the eigenvalues obtained from two calculations using **SIESTA**:

1. Scalar relativistic (spin-polarized)
2. Fully relativistic (spin–orbit coupling included)

---

### **Case 1 — Scalar relativistic (Spin-polarized)**

Example eigenvalues:-6.3407 -6.3407 -6.3407 -6.3367 -6.3367


#### **Observation**
- 5 Fe 3d levels split into **two groups**
- **3 nearly degenerate + 2 nearly degenerate**

#### **Interpretation**
This corresponds to **crystal-field splitting**:

$[
5d \rightarrow t_{2g}(3) + e_g(2)
]$

- cubic symmetry partially lifts degeneracy
- spin and orbital motion remain independent

Fermi energy:
$[
E_F = -2.768 \text{ eV}
]$

---

### **Case 2 — Fully relativistic (Spin–Orbit Coupling)**

Example eigenvalues:-6.3853 -6.3637 -6.3612 -6.3230 -6.2394


#### **Observation**
- all 5 energies are different
- no grouping remains

#### **Interpretation**

Spin–orbit coupling adds:

$[
H_{SOC} = \xi \mathbf{L}\cdot\mathbf{S}
]$

This:
- couples spin and orbital motion
- lowers symmetry further
- completely removes degeneracy

$[
5d \rightarrow 5 \text{ distinct levels}
]$

Fermi energy:
$[
E_F = -2.781 \text{ eV}
]$

---

## ***Comparison Summary***

| Feature | Scalar relativistic | With SOC |
|------------|-------------------|-----------|
| Degeneracy | 3 + 2 grouping | fully lifted |
| Splitting origin | crystal field only | crystal field + L·S coupling |
| Spin–orbit mixing | absent | present |
| Fermi energy | −2.768 eV | −2.781 eV |

---

## ***Final Conclusion***

- Without SOC: Fe 3d orbitals split only by the crystal field → **t₂g + e_g**
- With SOC: spin and orbital angular momentum mix → **complete splitting of all levels**
- SOC slightly shifts energies and modifies the electronic structure

Hence, SOC breaks the remaining degeneracy and produces a more accurate, fully relativistic band structure.






In [1]:
!pwd

/home/l-rishiraj/siesta-docs/work-files/tutorials/basic/magnetism/03_Fe_isolated


In [2]:
%cd EX1

/home/l-rishiraj/siesta-docs/work-files/tutorials/basic/magnetism/03_Fe_isolated/EX1


/home/l-rishiraj/miniconda3/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]
